# Chapter 2.6: Multi-Task Learning for RecSys

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why **multi-task learning** is essential for industrial recommendation systems
2. Implement the **Shared-Bottom** architecture and understand its limitations
3. Build **MMoE** (Multi-gate Mixture-of-Experts, Google 2018) from scratch
4. Implement **PLE** (Progressive Layered Extraction, Tencent 2020) for handling task conflicts
5. Understand **ESMM** (Alibaba, 2018) for CVR prediction using click data
6. Overview **AdaSparse** (Alibaba) and **EPNet** (Tencent)
7. Compare multi-task architectures on synthetic multi-objective data

## Prerequisites

- Chapters 2.1-2.5 (CTR prediction fundamentals and deep models)
- Understanding of multi-task learning basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part2/chapter_2.6_multi_task.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part2/chapter_2.6_multi_task.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

np.random.seed(42)
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

device = torch.device('cpu')

## 1. Why Multi-Task Learning in RecSys?

Industrial recommendation systems optimize for **multiple objectives** simultaneously:

| Task | Metric | Example |
|------|--------|---------|
| Click prediction (CTR) | P(click) | Will the user click this ad? |
| Conversion prediction (CVR) | P(purchase\|click) | Will the user buy after clicking? |
| Engagement | Watch time, likes | Will the user engage deeply? |
| Satisfaction | Dwell time, shares | Is the user satisfied? |

Multi-task learning (MTL) helps because:
1. **Shared representations** improve data efficiency
2. **Auxiliary tasks** act as regularizers
3. **Joint optimization** aligns feature learning across objectives

> **Common Pitfall:** Naive parameter sharing can cause **task conflicts** -- optimizing for one task degrades another. This is called the "seesaw effect" and motivated architectures like MMoE and PLE.

## 2. Generating Multi-Task CTR Data

In [ ]:
def generate_mtl_data(n_samples=30000, seed=42):
    """
    Generate data with two correlated but different tasks:
    Task A: Click prediction (CTR)
    Task B: Conversion prediction (CVR)
    """
    rng = np.random.RandomState(seed)
    
    field_dims = [10, 5, 15, 20, 8]
    n_fields = len(field_dims)
    total_features = sum(field_dims)
    offsets = np.array([0] + list(np.cumsum(field_dims[:-1])))
    
    data = np.zeros((n_samples, n_fields), dtype=np.int64)
    for f in range(n_fields):
        data[:, f] = rng.randint(0, field_dims[f], n_samples)
    global_data = data + offsets[np.newaxis, :]
    
    # Shared latent factors
    V_shared = rng.randn(total_features, 8) * 0.3
    
    # Task-specific factors
    V_taskA = rng.randn(total_features, 4) * 0.2
    V_taskB = rng.randn(total_features, 4) * 0.2
    
    logits_A = np.zeros(n_samples)
    logits_B = np.zeros(n_samples)
    
    for i in range(n_samples):
        feats = global_data[i]
        
        # Shared component
        shared = np.sum(V_shared[feats], axis=0)
        shared_score = 0.3 * np.sum(shared ** 2)
        
        # Task-specific components
        taskA_vec = np.sum(V_taskA[feats], axis=0)
        taskB_vec = np.sum(V_taskB[feats], axis=0)
        
        logits_A[i] = shared_score + 0.5 * np.sum(taskA_vec ** 2) - 2.0
        logits_B[i] = shared_score + 0.5 * np.sum(taskB_vec ** 2) - 3.0
    
    # Add some negative correlation between tasks (task conflict)
    conflict = rng.randn(n_samples) * 0.3
    logits_A += conflict
    logits_B -= 0.5 * conflict
    
    probs_A = 1.0 / (1.0 + np.exp(-np.clip(logits_A, -10, 10)))
    probs_B = 1.0 / (1.0 + np.exp(-np.clip(logits_B, -10, 10)))
    
    labels_A = rng.binomial(1, probs_A)
    labels_B = rng.binomial(1, probs_B)
    
    print(f"Generated {n_samples} samples")
    print(f"Task A (CTR): {labels_A.mean():.4f}")
    print(f"Task B (CVR): {labels_B.mean():.4f}")
    print(f"Correlation: {np.corrcoef(labels_A, labels_B)[0,1]:.4f}")
    
    return global_data, labels_A, labels_B, field_dims, offsets

data, labels_A, labels_B, field_dims, offsets = generate_mtl_data()
total_features = sum(field_dims)
n_fields = len(field_dims)

split = 24000
X_train, X_test = data[:split], data[split:]
yA_train, yA_test = labels_A[:split], labels_A[split:]
yB_train, yB_test = labels_B[:split], labels_B[split:]

## 3. Shared-Bottom Architecture

The simplest MTL approach: share all bottom layers, use separate tower networks for each task.

$$ \mathbf{h} = f_{\text{shared}}(\mathbf{x}) $$
$$ \hat{y}_A = \text{tower}_A(\mathbf{h}), \quad \hat{y}_B = \text{tower}_B(\mathbf{h}) $$

> **Common Pitfall:** With conflicting tasks, shared-bottom suffers from "negative transfer" -- one task's gradients can hurt the other.

In [ ]:
class SharedBottom(nn.Module):
    """Shared-Bottom multi-task model."""
    
    def __init__(self, num_features, n_fields, embedding_dim=8,
                 shared_dims=None, tower_dims=None):
        super().__init__()
        if shared_dims is None:
            shared_dims = [128, 64]
        if tower_dims is None:
            tower_dims = [32]
        
        self.embedding = nn.Embedding(num_features, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        
        # Shared bottom
        input_dim = n_fields * embedding_dim
        shared_layers = []
        for h in shared_dims:
            shared_layers.extend([nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(0.1)])
            input_dim = h
        self.shared = nn.Sequential(*shared_layers)
        
        # Task towers
        self.tower_A = self._make_tower(shared_dims[-1], tower_dims)
        self.tower_B = self._make_tower(shared_dims[-1], tower_dims)
    
    def _make_tower(self, input_dim, tower_dims):
        layers = []
        for h in tower_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU()])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        embed = self.embedding(x).view(x.size(0), -1)
        shared_out = self.shared(embed)
        return self.tower_A(shared_out).squeeze(-1), self.tower_B(shared_out).squeeze(-1)

print("SharedBottom model defined.")

## 4. MMoE: Multi-gate Mixture-of-Experts (Google, 2018)

MMoE addresses task conflicts by using **multiple expert networks** and **task-specific gating**:

$$ f^k(\mathbf{x}) = \sum_{i=1}^{n} g^k_i(\mathbf{x}) \cdot f_i(\mathbf{x}) $$

where:
- $f_i(\mathbf{x})$ is expert $i$'s output
- $g^k(\mathbf{x}) = \text{softmax}(\mathbf{W}^k_g \mathbf{x})$ is task $k$'s gating network

> **Concept:** Each task has its own gating network that decides which experts to rely on. Correlated tasks will have similar gating patterns; conflicting tasks will use different experts.

In [ ]:
class MMoE(nn.Module):
    """Multi-gate Mixture-of-Experts (Google, 2018)."""
    
    def __init__(self, num_features, n_fields, embedding_dim=8,
                 n_experts=4, expert_dim=64, n_tasks=2, tower_dims=None):
        super().__init__()
        if tower_dims is None:
            tower_dims = [32]
        
        self.n_experts = n_experts
        self.n_tasks = n_tasks
        
        self.embedding = nn.Embedding(num_features, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        
        input_dim = n_fields * embedding_dim
        
        # Expert networks
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, expert_dim),
                nn.ReLU()
            )
            for _ in range(n_experts)
        ])
        
        # Task-specific gating networks
        self.gates = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, n_experts),
                nn.Softmax(dim=-1)
            )
            for _ in range(n_tasks)
        ])
        
        # Task towers
        self.towers = nn.ModuleList([
            self._make_tower(expert_dim, tower_dims)
            for _ in range(n_tasks)
        ])
    
    def _make_tower(self, input_dim, tower_dims):
        layers = []
        for h in tower_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU()])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        embed = self.embedding(x).view(x.size(0), -1)  # (batch, input_dim)
        
        # Expert outputs
        expert_outputs = [expert(embed) for expert in self.experts]  # list of (batch, expert_dim)
        expert_stack = torch.stack(expert_outputs, dim=1)  # (batch, n_experts, expert_dim)
        
        # Task outputs
        task_outputs = []
        for task_id in range(self.n_tasks):
            gate_weights = self.gates[task_id](embed)  # (batch, n_experts)
            # Weighted sum of expert outputs
            gate_out = (expert_stack * gate_weights.unsqueeze(-1)).sum(dim=1)  # (batch, expert_dim)
            task_out = self.towers[task_id](gate_out).squeeze(-1)  # (batch,)
            task_outputs.append(task_out)
        
        return task_outputs[0], task_outputs[1]

print("MMoE model defined.")
mmoe = MMoE(total_features, n_fields)
print(f"MMoE parameters: {sum(p.numel() for p in mmoe.parameters())}")

## 5. PLE: Progressive Layered Extraction (Tencent, 2020)

PLE improves over MMoE by having **task-specific experts** in addition to shared experts, and using **progressive extraction** across multiple levels.

At each extraction level:
- **Shared experts**: process shared representations
- **Task-specific experts**: each task has its own set of experts
- **Extraction gates**: each task's gate selects from ALL experts (shared + task-specific)

> **Pro Tip:** PLE was shown to outperform MMoE particularly when tasks have complex relationships (partially correlated, partially conflicting). It's the default MTL architecture at Tencent's recommendation systems.

In [ ]:
class PLELayer(nn.Module):
    """Single PLE extraction layer."""
    
    def __init__(self, input_dim, expert_dim, n_shared_experts=2, 
                 n_task_experts=2, n_tasks=2):
        super().__init__()
        self.n_tasks = n_tasks
        self.n_shared = n_shared_experts
        self.n_task = n_task_experts
        total_experts = n_shared_experts + n_task_experts * n_tasks
        
        # Shared experts
        self.shared_experts = nn.ModuleList([
            nn.Sequential(nn.Linear(input_dim, expert_dim), nn.ReLU())
            for _ in range(n_shared_experts)
        ])
        
        # Task-specific experts
        self.task_experts = nn.ModuleList([
            nn.ModuleList([
                nn.Sequential(nn.Linear(input_dim, expert_dim), nn.ReLU())
                for _ in range(n_task_experts)
            ])
            for _ in range(n_tasks)
        ])
        
        # Gates for each task
        # Each task's gate selects from shared + its own task experts
        n_experts_per_task = n_shared_experts + n_task_experts
        self.gates = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, n_experts_per_task),
                nn.Softmax(dim=-1)
            )
            for _ in range(n_tasks)
        ])
    
    def forward(self, x):
        """x: (batch, input_dim). Returns list of (batch, expert_dim) per task."""
        # Shared expert outputs
        shared_outs = [expert(x) for expert in self.shared_experts]
        
        task_outputs = []
        for task_id in range(self.n_tasks):
            # Task-specific expert outputs
            task_outs = [expert(x) for expert in self.task_experts[task_id]]
            
            # Stack all experts for this task: shared + task-specific
            all_outs = shared_outs + task_outs
            expert_stack = torch.stack(all_outs, dim=1)  # (batch, n_experts, expert_dim)
            
            # Gating
            gate_w = self.gates[task_id](x)  # (batch, n_experts)
            gated = (expert_stack * gate_w.unsqueeze(-1)).sum(dim=1)  # (batch, expert_dim)
            task_outputs.append(gated)
        
        return task_outputs


class PLE(nn.Module):
    """Progressive Layered Extraction (Tencent, 2020)."""
    
    def __init__(self, num_features, n_fields, embedding_dim=8,
                 n_ple_layers=2, expert_dim=64, n_shared_experts=2,
                 n_task_experts=2, n_tasks=2, tower_dims=None):
        super().__init__()
        if tower_dims is None:
            tower_dims = [32]
        
        self.n_tasks = n_tasks
        
        self.embedding = nn.Embedding(num_features, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        
        input_dim = n_fields * embedding_dim
        
        # PLE layers
        self.ple_layers = nn.ModuleList()
        for i in range(n_ple_layers):
            in_d = input_dim if i == 0 else expert_dim
            self.ple_layers.append(
                PLELayer(in_d, expert_dim, n_shared_experts, 
                         n_task_experts, n_tasks)
            )
        
        # Task towers
        self.towers = nn.ModuleList([
            self._make_tower(expert_dim, tower_dims)
            for _ in range(n_tasks)
        ])
    
    def _make_tower(self, input_dim, tower_dims):
        layers = []
        for h in tower_dims:
            layers.extend([nn.Linear(input_dim, h), nn.ReLU()])
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        embed = self.embedding(x).view(x.size(0), -1)
        
        # Pass through PLE layers
        task_reps = None
        for ple_layer in self.ple_layers:
            if task_reps is None:
                task_reps = ple_layer(embed)
            else:
                # Average task representations as input to next layer
                task_reps = [ple_layer(rep) for rep in task_reps]
                task_reps = [task_reps[t][t] for t in range(self.n_tasks)]
                # Re-apply PLE
                # Simplified: use first task's output
        
        # This simplified version uses a single PLE application
        # For proper multi-level PLE, you'd chain them more carefully
        if isinstance(task_reps[0], list):
            task_reps = [task_reps[t][t] for t in range(self.n_tasks)]
        
        outputs = []
        for t in range(self.n_tasks):
            outputs.append(self.towers[t](task_reps[t]).squeeze(-1))
        
        return outputs[0], outputs[1]

print("PLE model defined.")

## 6. Training Multi-Task Models

In [ ]:
def train_mtl_model(model, X_train, yA_train, yB_train,
                    X_test, yA_test, yB_test,
                    epochs=25, batch_size=256, lr=0.001):
    """Train a multi-task model."""
    train_ds = TensorDataset(
        torch.LongTensor(X_train),
        torch.FloatTensor(yA_train),
        torch.FloatTensor(yB_train)
    )
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    
    history = {'train_loss': [], 'aucA': [], 'aucB': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0.0, 0
        for bx, byA, byB in loader:
            optimizer.zero_grad()
            outA, outB = model(bx)
            lossA = criterion(outA, byA)
            lossB = criterion(outB, byB)
            loss = lossA + lossB  # Equal weighting
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        
        model.eval()
        with torch.no_grad():
            tx = torch.LongTensor(X_test)
            pA, pB = model(tx)
            probsA = torch.sigmoid(pA).numpy()
            probsB = torch.sigmoid(pB).numpy()
            aucA = roc_auc_score(yA_test, probsA)
            aucB = roc_auc_score(yB_test, probsB)
        
        history['train_loss'].append(total_loss / n_batches)
        history['aucA'].append(aucA)
        history['aucB'].append(aucB)
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}: loss={total_loss/n_batches:.4f}, "
                  f"AUC_A={aucA:.4f}, AUC_B={aucB:.4f}")
    
    return history

# Train all models
print("=== Shared-Bottom ===")
torch.manual_seed(42)
sb_model = SharedBottom(total_features, n_fields)
sb_hist = train_mtl_model(sb_model, X_train, yA_train, yB_train,
                          X_test, yA_test, yB_test)

print("\n=== MMoE ===")
torch.manual_seed(42)
mmoe_model = MMoE(total_features, n_fields, n_experts=4)
mmoe_hist = train_mtl_model(mmoe_model, X_train, yA_train, yB_train,
                            X_test, yA_test, yB_test)

print("\n=== PLE ===")
torch.manual_seed(42)
ple_model = PLE(total_features, n_fields, n_ple_layers=1)
ple_hist = train_mtl_model(ple_model, X_train, yA_train, yB_train,
                           X_test, yA_test, yB_test)

In [ ]:
# Comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, 26)

# Task A AUC
axes[0].plot(epochs, sb_hist['aucA'], 'b-o', label='Shared-Bottom', linewidth=2, markersize=3)
axes[0].plot(epochs, mmoe_hist['aucA'], 'r-s', label='MMoE', linewidth=2, markersize=3)
axes[0].plot(epochs, ple_hist['aucA'], 'g-^', label='PLE', linewidth=2, markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].set_title('Task A (CTR) AUC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Task B AUC
axes[1].plot(epochs, sb_hist['aucB'], 'b-o', label='Shared-Bottom', linewidth=2, markersize=3)
axes[1].plot(epochs, mmoe_hist['aucB'], 'r-s', label='MMoE', linewidth=2, markersize=3)
axes[1].plot(epochs, ple_hist['aucB'], 'g-^', label='PLE', linewidth=2, markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Task B (CVR) AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Final comparison bar chart
models = ['Shared-Bot', 'MMoE', 'PLE']
final_A = [sb_hist['aucA'][-1], mmoe_hist['aucA'][-1], ple_hist['aucA'][-1]]
final_B = [sb_hist['aucB'][-1], mmoe_hist['aucB'][-1], ple_hist['aucB'][-1]]

x = np.arange(len(models))
width = 0.35
axes[2].bar(x - width/2, final_A, width, label='Task A', color='steelblue', edgecolor='black')
axes[2].bar(x + width/2, final_B, width, label='Task B', color='coral', edgecolor='black')
axes[2].set_xticks(x)
axes[2].set_xticklabels(models)
axes[2].set_ylabel('AUC')
axes[2].set_title('Final AUC Comparison')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. ESMM: Entire Space Multi-Task Model (Alibaba, 2018)

ESMM addresses the **CVR sample selection bias**: we only observe conversions for items that were clicked, but we want to predict conversion for ALL items.

Key insight: decompose CTCVR (click AND convert) as:

$$ P(\text{conversion, click}) = P(\text{click}) \times P(\text{conversion} | \text{click}) $$

$$ \text{CTCVR} = \text{CTR} \times \text{CVR} $$

Both CTR and CVR towers are trained over the **entire sample space** (not just clicked samples), eliminating selection bias.

> **Concept:** ESMM never directly trains CVR on clicked-only data. Instead, it multiplies CTR and CVR predictions to get CTCVR, and trains the CTCVR output on all impression data.

In [ ]:
class ESMM(nn.Module):
    """Entire Space Multi-Task Model (Alibaba, 2018)."""
    
    def __init__(self, num_features, n_fields, embedding_dim=8, hidden_dims=None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [64, 32]
        
        self.embedding = nn.Embedding(num_features, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        
        input_dim = n_fields * embedding_dim
        
        # CTR tower
        ctr_layers = []
        dim = input_dim
        for h in hidden_dims:
            ctr_layers.extend([nn.Linear(dim, h), nn.ReLU(), nn.Dropout(0.1)])
            dim = h
        ctr_layers.append(nn.Linear(dim, 1))
        self.ctr_tower = nn.Sequential(*ctr_layers)
        
        # CVR tower
        cvr_layers = []
        dim = input_dim
        for h in hidden_dims:
            cvr_layers.extend([nn.Linear(dim, h), nn.ReLU(), nn.Dropout(0.1)])
            dim = h
        cvr_layers.append(nn.Linear(dim, 1))
        self.cvr_tower = nn.Sequential(*cvr_layers)
    
    def forward(self, x):
        embed = self.embedding(x).view(x.size(0), -1)
        
        ctr_logit = self.ctr_tower(embed).squeeze(-1)
        cvr_logit = self.cvr_tower(embed).squeeze(-1)
        
        ctr_prob = torch.sigmoid(ctr_logit)
        cvr_prob = torch.sigmoid(cvr_logit)
        
        # CTCVR = CTR * CVR
        ctcvr_prob = ctr_prob * cvr_prob
        
        return ctr_prob, cvr_prob, ctcvr_prob

print("ESMM model defined.")
print("Key feature: CVR trained over entire space via CTCVR decomposition.")

## 8. MMoE Expert Gate Analysis

In [ ]:
# Analyze which experts each task relies on
mmoe_model.eval()
with torch.no_grad():
    tx = torch.LongTensor(X_test[:1000])
    embed = mmoe_model.embedding(tx).view(tx.size(0), -1)
    
    gate_A = mmoe_model.gates[0](embed).numpy()  # (1000, n_experts)
    gate_B = mmoe_model.gates[1](embed).numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Average gate weights
avg_gate_A = gate_A.mean(axis=0)
avg_gate_B = gate_B.mean(axis=0)

x_pos = np.arange(4)
width = 0.35
axes[0].bar(x_pos - width/2, avg_gate_A, width, label='Task A (CTR)', color='steelblue')
axes[0].bar(x_pos + width/2, avg_gate_B, width, label='Task B (CVR)', color='coral')
axes[0].set_xlabel('Expert ID')
axes[0].set_ylabel('Average Gate Weight')
axes[0].set_title('MMoE: Expert Utilization by Task')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Gate weight distribution for expert 0
axes[1].hist(gate_A[:, 0], bins=30, alpha=0.6, label='Task A', color='steelblue')
axes[1].hist(gate_B[:, 0], bins=30, alpha=0.6, label='Task B', color='coral')
axes[1].set_xlabel('Gate Weight for Expert 0')
axes[1].set_ylabel('Count')
axes[1].set_title('Gate Weight Distribution (Expert 0)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gate weight correlation between tasks
for i in range(4):
    axes[2].scatter(gate_A[:, i], gate_B[:, i], alpha=0.1, s=5, label=f'Expert {i}')
axes[2].set_xlabel('Task A Gate Weight')
axes[2].set_ylabel('Task B Gate Weight')
axes[2].set_title('Gate Weight Correlation Between Tasks')
axes[2].legend(markerscale=5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Implement MMoE from Scratch

In [ ]:
# Exercise 1: Implement MMoE with configurable number of tasks
# TODO: Create an MMoE that works for 3 tasks instead of 2
# TODO: Generate 3-task synthetic data
# TODO: Train and evaluate on all three tasks
# TODO: Visualize the expert utilization across 3 tasks

# YOUR CODE HERE

### Exercise 2: Task Conflict Analysis

In [ ]:
# Exercise 2: How do task relationships affect MTL performance?
# TODO: Generate 3 datasets with different task relationships:
#   a) Highly correlated tasks (correlation ~ 0.8)
#   b) Weakly correlated tasks (correlation ~ 0.3)
#   c) Conflicting tasks (correlation ~ -0.3)
# TODO: Train Shared-Bottom, MMoE, and PLE on each
# TODO: Which architecture is most robust to task conflicts?

# YOUR CODE HERE

### Exercise 3: Implement PLE with Multiple Extraction Levels

In [ ]:
# Exercise 3: Full PLE with 2-3 extraction levels
# TODO: Implement proper multi-level PLE where:
#   - Each level's task input is the previous level's gated output for that task
#   - Shared experts also get updated across levels
# TODO: Compare 1-level vs 2-level vs 3-level PLE
# TODO: Does more levels always help?

# YOUR CODE HERE

### Exercise 4: Loss Weighting Strategies

In [ ]:
# Exercise 4: Experiment with different loss weighting strategies
# TODO: Implement and compare:
#   a) Equal weighting: L = L_A + L_B
#   b) Fixed weighting: L = w_A * L_A + w_B * L_B
#   c) Uncertainty weighting (Kendall et al., 2018):
#      L = (1/2*sigma_A^2) * L_A + (1/2*sigma_B^2) * L_B + log(sigma_A) + log(sigma_B)
# TODO: Which strategy gives the best combined performance?

# YOUR CODE HERE

## Summary

In this chapter, we covered:

1. **Shared-Bottom**: Simple parameter sharing, but suffers from task conflicts
2. **MMoE** (Google, 2018): Multiple experts with task-specific gating to handle different task needs
3. **PLE** (Tencent, 2020): Task-specific + shared experts with progressive extraction
4. **ESMM** (Alibaba, 2018): CVR prediction over entire sample space via CTCVR decomposition
5. **AdaSparse** and **EPNet**: Advanced techniques for adaptive sparsity and explicit preference networks

### Key Takeaways

- Multi-task learning is essential in production RecSys for optimizing multiple business metrics
- Task conflicts are real and need architectural solutions (not just loss weighting)
- MMoE's gating mechanism allows different tasks to use different expert combinations
- PLE's task-specific experts further reduce negative transfer
- ESMM elegantly solves the CVR sample selection bias problem

### What's Next

In Chapter 2.7, we'll explore **advanced feature interaction** methods including AutoInt, FiBiNET, and MaskNet.